# Demo: Generating Text One Token at a Time

This notebook provides a brief demonstration of how a large language model (LLM) generates text. We will walk through the process of loading a model, tokenizing text, and generating new text step-by-step.

### Step 1. Load a Tokenizer and a Model

First, we need to load a pre-trained model and its corresponding tokenizer from the Hugging Face `transformers` library. The **tokenizer** converts text into a sequence of numbers (tokens) that the model can process, and the **model** will perform the text generation.

Downloads the model at **C:\Users<your-username>.cache\huggingface\hub**

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# For this demo, we'll use 'distilgpt2', a smaller and faster version of GPT-2
# Creates via knowledge distillation, retaining much of the original model's capabilities while being more efficient
model_name = "distilgpt2"

# Load the tokenizer and model associated with our chosen model name
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [2]:
# View model architecture
print(model)

# View model params
print(f"Total parameters in the model: {sum(p.numel() for p in model.parameters()):,}")

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)
Total parameters in the model: 81,912,576


In [3]:
# View tokenizer details
print(tokenizer)

GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


### Step 2. Examine the Tokenization

Let's see how the tokenizer converts a simple sentence into a list of token IDs. This process is called **tokenization**.

In [4]:
# Define a starting phrase, also known as a prompt
prompt_text = "Machine learning is a field of"

# Use the tokenizer to convert the text prompt into input tensors for the model
inputs = tokenizer(prompt_text, return_tensors="pt")

print(type(inputs), dict(inputs).keys())

# The 'input_ids' are the numerical representations of our text
print("Prompt text:", prompt_text)
print("Token IDs:", inputs["input_ids"])
print("Token IDs shape:", inputs["input_ids"].shape)
print("Attention mask:", inputs["attention_mask"])

<class 'transformers.tokenization_utils_base.BatchEncoding'> dict_keys(['input_ids', 'attention_mask'])
Prompt text: Machine learning is a field of
Token IDs: tensor([[37573,  4673,   318,   257,  2214,   286]])
Token IDs shape: torch.Size([1, 6])
Attention mask: tensor([[1, 1, 1, 1, 1, 1]])


To better understand the tokenization, we can decode each ID back into its corresponding text. Notice that some tokens are whole words, while others are parts of words or punctuation. This is called **subword tokenization**.

In [5]:
import pandas as pd

# Get the list of token IDs from our inputs
token_ids = inputs["input_ids"][0].tolist()
print(f"token_ids: {token_ids}")

# Decode each token ID back to its string representation
tokens = [tokenizer.decode(token_id) for token_id in token_ids]

# Display the IDs and their corresponding tokens in a table for clarity
token_df = pd.DataFrame(data={"ID": token_ids, "Token": tokens})

print("========================================================")
print(token_df.head())

token_ids: [37573, 4673, 318, 257, 2214, 286]
      ID      Token
0  37573    Machine
1   4673   learning
2    318         is
3    257          a
4   2214      field


### Step 3. Generate the Next Token

Now, we'll feed our tokenized prompt to the model and ask it to predict the single most likely next token.

**Outputs.logits** shape will be `[batch_size, sequence_length, vocab_size]`

In [6]:
# We use torch.no_grad() to disable gradient calculations, as we are not training the model
with torch.no_grad():
    # Get the model's raw output, called 'logits'
    outputs = model(**inputs)
    print(type(outputs), outputs.keys())
    print(outputs.logits.shape)
    print("============================")
    # We only care about the logits for the very last token in our input sequence
    next_token_logits = outputs.logits[:, -1, :]

    # Convert logits into probabilities using the softmax function
    probabilities = torch.nn.functional.softmax(next_token_logits, dim=-1)

    # Find the token ID with the highest probability
    most_likely_next_token_id = torch.argmax(probabilities).item()

print(f"The most likely next token ID is: {most_likely_next_token_id}")
print(f"This token is: '{tokenizer.decode(most_likely_next_token_id)}'")

<class 'transformers.modeling_outputs.CausalLMOutputWithCrossAttentions'> odict_keys(['logits', 'past_key_values'])
torch.Size([1, 6, 50257])
The most likely next token ID is: 2267
This token is: ' research'


In [7]:
# All logits for the last token in the input sequence
last_logits = outputs.logits[0][-1]
print(f"Logits shape: {last_logits.shape}")
# Get the top 5 most likely next tokens and their probabilities
top_k = 5   
top_k_logits, top_k_indices = torch.topk(last_logits, top_k)
print(top_k_logits, top_k_indices)
print(tokenizer.decode(2267))

Logits shape: torch.Size([50257])
tensor([-60.9906, -61.2284, -62.3265, -62.3823, -62.6386]) tensor([ 2267,  2050,  1807,  3783, 13572])
 research


By repeatedly predicting the next token and appending it to our input, we can generate a longer sequence of text.

In [8]:
# Let's generate a few more tokens by repeating the process in a loop
generated_ids = inputs["input_ids"]

print("Generating 5 tokens one at a time:")
print(tokenizer.decode(generated_ids[0]), end="")
print("... ", end="")  # Indicate that we're generating more text
# This loop generates one token at a time
for _ in range(5):
    with torch.no_grad():
        outputs = model(generated_ids)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

    # Append the newly predicted token ID to our sequence
    generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

    # Print the newly generated token
    print(tokenizer.decode(next_token_id[0]), end="")

Generating 5 tokens one at a time:
Machine learning is a field of...  research that has been growing

### Step 4. Use the `generate` Method

Generating tokens one-by-one manually is great for understanding the process, but it's inefficient. The `transformers` library provides a convenient `.generate()` method that handles this entire process for us.

In [9]:
tokenizer.eos_token_id, tokenizer.decode(tokenizer.eos_token_id)

(50256, '<|endoftext|>')

In [10]:
from IPython.display import Markdown, display

# We start with the same tokenized prompt
inputs = tokenizer(prompt_text, return_tensors="pt")

# Use the .generate() method to create a sequence of a desired length
# Generate up to 50 tokens in total (including the prompt), and use the EOS token ID for padding if needed  
output_ids = model.generate(
    **inputs, max_length=50, pad_token_id=tokenizer.eos_token_id
)

# Decode the entire sequence of token IDs into a single string
generated_text = tokenizer.decode(output_ids[0])

print("--- Text Generated with model.generate() ---")
display(Markdown(generated_text))

--- Text Generated with model.generate() ---


Machine learning is a field of research that has been growing in the past few years.


































This demo shows the core logic of how a language model generates text. Now you're ready to try it yourself!

<br /><br /><br /><br /><br /><br /><br /><br /><br />